# Pelatihan dan evaluasi GSTL di Google Colab

Melatih GSTL (protokol `REPLIKASI.md`) dan mengevaluasi hasilnya. Semua keluaran disimpan ke Google Drive Anda di `Research/gstl_replication/hasil/<EXP>/` dan dapat **dilanjutkan** bila sesi Colab terputus.

**Sebelum mulai**
1. *Runtime → Change runtime type → GPU.*
2. Repo privat? Simpan token GitHub (hanya baca) di *Secrets* dengan nama `GITHUB_TOKEN`, aktifkan *Notebook access*.
3. **Mode `sintetis`** (bawaan): data sintetis kecil untuk memastikan notebook berjalan; bukan hasil riset. **Mode `ccd`**: butuh data CCD asli di Drive (lihat sel 4).

**Peringatan jujur**
- Fitur CCD (`vgg16_features/`) ±60 GB. Colab gratis dan Drive gratis (15 GB) umumnya **tidak cukup**; butuh Drive berbayar dan disk Colab besar. Sel 5 memeriksa ruang dan berhenti dengan pesan jelas bila kurang.
- Di RTX 3060 satu seed ±2,4 jam dengan early stopping. Kecepatan di GPU Colab **belum diukur**.
- Sesi gratis bisa putus kapan saja. Notebook menyalin keluaran ke Drive tiap 5 menit dan melanjutkan dari `snapshot/last.pth` saat dijalankan ulang dengan `EXP` yang sama.

In [ ]:
# 1. Periksa GPU dan versi
import sys, torch, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", numpy.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA (pelatihan di CPU sangat lambat)")

In [ ]:
# 2. Ambil kode dari GitHub (token dari Secrets bila ada; tidak pernah dicetak)
import os, subprocess
REPO = "taufiqmus/gstl_replication"
if not os.path.isdir("/content/gstl_replication"):
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        tok = None
        print("Token tidak terbaca dari Secrets (%s). Cek: nama persis GITHUB_TOKEN, nilai terisi, dan sakelar 'Notebook access' menyala." % type(e).__name__)
    url = ("https://%s@github.com/%s.git" % (tok, REPO)) if tok else ("https://github.com/%s.git" % REPO)
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, "/content/gstl_replication"], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("clone gagal (repo privat tanpa token, atau token salah): " + r.stderr.replace(tok or "<none>", "***"))
    if tok:  # hapus token dari konfigurasi git di mesin Colab
        subprocess.run(["git", "-C", "/content/gstl_replication", "remote", "set-url", "origin", "https://github.com/%s.git" % REPO])
os.chdir("/content/gstl_replication")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Drive dan fungsi bantu (jalankan perintah, sinkron ke Drive)
import os, shutil, json, subprocess, sys, threading, time
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Research/gstl_replication"
HASIL_DIR = DRIVE_DIR + "/hasil"
os.makedirs(HASIL_DIR, exist_ok=True)

def sh(cmd):
    """Jalankan perintah, tampilkan keluaran langsung; kembalikan kode keluar."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def sinkron(src, dst):
    """Salin berkas baru/berubah dari src ke dst (tidak menghapus apa pun di dst)."""
    n = 0
    for r, _, fs in os.walk(src):
        d = os.path.join(dst, os.path.relpath(r, src))
        os.makedirs(d, exist_ok=True)
        for f in fs:
            a, b = os.path.join(r, f), os.path.join(d, f)
            if not os.path.exists(b) or os.path.getsize(a) != os.path.getsize(b) or os.path.getmtime(a) > os.path.getmtime(b) + 1:
                shutil.copy2(a, b)
                n += 1
    return n

In [ ]:
# 4. KONFIGURASI (ubah di sini)
MODE = "sintetis"          # "sintetis" (uji notebook) atau "ccd" (data asli)
EXP = "gstl_s123"          # nama eksperimen -> hasil/<EXP>/ ; nama sama = lanjutkan dari last.pth
SEED = 123
EPOCHS = 70                # paper: 70 (early stopping biasanya berhenti ~epoch 20)
SEEDS_EVAL = "123"         # seed sampling BNN untuk evaluasi, mis. "123 1 2 3 4 5"
DATA_DRIVE = DRIVE_DIR + "/ccd"   # mode ccd: folder berisi vgg16_features/ dan videos/Crash-1500.txt
COPY_LOCAL = True          # mode ccd: salin data ke disk Colab (cepat); False = baca langsung dari Drive (sangat lambat)
MAX_TRAIN_ITERS = 0        # 0 = penuh; isi mis. 5 untuk uji sangat cepat
if MODE == "sintetis":
    EPOCHS, MAX_TRAIN_ITERS = min(EPOCHS, 3), (MAX_TRAIN_ITERS or 5)
print(MODE, EXP, "epochs", EPOCHS)

In [ ]:
# 5. Siapkan data
if MODE == "sintetis":
    DATA = "/content/data_sintetis"
    if not os.path.isdir(DATA):
        assert sh("python tools/make_synthetic_ccd.py --out %s --n-pos 20 --n-neg 40" % DATA) == 0
else:
    for need in ("vgg16_features", "videos/Crash-1500.txt"):
        if not os.path.exists(os.path.join(DATA_DRIVE, need)):
            raise SystemExit("tidak ada %s di %s" % (need, DATA_DRIVE))
    if COPY_LOCAL:
        DATA = "/content/ccd"
        ukuran = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(DATA_DRIVE + "/vgg16_features") for f in fs)
        bebas = shutil.disk_usage("/content").free
        print("fitur: %.1f GB | ruang bebas Colab: %.1f GB" % (ukuran / 1e9, bebas / 1e9))
        if ukuran * 1.05 > bebas:
            raise SystemExit("ruang disk Colab tidak cukup; set COPY_LOCAL = False (lambat) atau kecilkan data")
        if not os.path.isdir(DATA + "/vgg16_features"):
            os.makedirs(DATA + "/videos", exist_ok=True)
            shutil.copy2(DATA_DRIVE + "/videos/Crash-1500.txt", DATA + "/videos/Crash-1500.txt")
            shutil.copytree(DATA_DRIVE + "/vgg16_features", DATA + "/vgg16_features")
    else:
        DATA = DATA_DRIVE
print("data:", DATA)

In [ ]:
# 6. Latih. Keluaran lokal -> disalin ke Drive tiap 5 menit; resume otomatis bila ada last.pth di Drive.
LOKAL_ROOT = "/content/hasil_lokal"
LOKAL_EXP = LOKAL_ROOT + "/" + EXP
DRIVE_EXP = HASIL_DIR + "/" + EXP
os.makedirs(LOKAL_ROOT, exist_ok=True)

resume = ""
if os.path.exists(DRIVE_EXP + "/snapshot/last.pth"):
    sinkron(DRIVE_EXP, LOKAL_EXP)
    resume = "--resume %s/snapshot/last.pth" % LOKAL_EXP
    print("melanjutkan dari Drive:", DRIVE_EXP)

cmd = ("python -W ignore -u train.py --dataset crash --data-root %s --exp %s --out-root %s --epochs %d "
       "--val-frac 0.1 --test-iter 0 --eval-epoch-end --select-metric AP_video --early-stop-patience 10 "
       "--keep-all-epochs --seed %d --max-train-iters %d %s") % (DATA, EXP, LOKAL_ROOT, EPOCHS, SEED, MAX_TRAIN_ITERS, resume)
print(cmd)

stop = threading.Event()
def _loop():
    while not stop.wait(300):
        if os.path.isdir(LOKAL_EXP):
            sinkron(LOKAL_EXP, DRIVE_EXP)
t = threading.Thread(target=_loop, daemon=True)
t.start()
kode = None
try:
    kode = sh(cmd)
finally:
    stop.set()
    t.join()
    if os.path.isdir(LOKAL_EXP):
        print("sinkron akhir ke Drive:", sinkron(LOKAL_EXP, DRIVE_EXP), "berkas")
print("kode keluar train.py:", kode)

In [ ]:
# 7. Evaluasi semua checkpoint epoch pada set uji (seed sampling di SEEDS_EVAL)
TAG = "uji_" + EXP
assert sh("python -W ignore -u evaluate.py --data-root %s --exp-dir %s --seeds %s --tag %s --out %s/evaluasi" % (DATA, LOKAL_EXP, SEEDS_EVAL, TAG, LOKAL_EXP)) == 0
print("evaluasi disalin:", sinkron(LOKAL_EXP + "/evaluasi", DRIVE_EXP + "/evaluasi"), "berkas")

In [ ]:
# 8. Ringkasan
import csv
print("hasil di Drive:", DRIVE_EXP)
for d in sorted(os.listdir(DRIVE_EXP)):
    p = os.path.join(DRIVE_EXP, d)
    n = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs) if os.path.isdir(p) else os.path.getsize(p)
    print("  %-30s %8.1f MB" % (d, n / 1e6))
csvs = [f for f in os.listdir(DRIVE_EXP + "/evaluasi") if f.endswith(".csv")]
if csvs:
    rows = list(csv.DictReader(open(DRIVE_EXP + "/evaluasi/" + csvs[0])))
    print("kolom:", list(rows[0].keys()))
    print("baris terakhir:", rows[-1])

## Catatan
- Mode `sintetis` memakai data buatan: angka metriknya **tidak bermakna** dan hanya membuktikan notebook berjalan.
- Hasil CCD satu seed belum punya selang kepercayaan; untuk 3-5 seed jalankan ulang dengan `SEED` dan `EXP` berbeda (mis. `gstl_s1`).
- Bobot GSTL yang dihasilkan memuat `config`; untuk dipakai di demo, salin `snapshot/best.pth` ke `weights/` dan beri `--weights`.
- Menjalankan ulang dengan `EXP` yang sama melanjutkan dari `snapshot/last.pth`; ganti `EXP` untuk mulai baru.